In [37]:
# Reload modules automatically
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
import cma
import numpy as np
from numbers import Real
import torch
from PIL import Image
import random
import requests
from io import BytesIO
import time
from matplotlib import cm
import matplotlib.pyplot as plt
from tqdm import tqdm
from os import path
import os
import json
from IPython.display import display, clear_output


# Utils imports
from utils.rasterize import *
from utils.load import get_segment_imgs, get_primitive_imgs
from utils.clipemb import CLIP_emb_from_IMG, CLIP_emb_from_TEXT
from utils.cma import clip_sol


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

WIDTH = 300
HEIGHT = 300

CONTENT_DIR = './data'
RESULTS_DIR = path.join(CONTENT_DIR, 'results')
SEGMENT_DIR = path.join(CONTENT_DIR, 'segment_img')
RW_PRIMITIVES_DIR = path.join(CONTENT_DIR, 'rw_primitives')
PRIMITIVE_DIR = path.join(CONTENT_DIR, 'primitives')

SEGMENT_IMGS = get_segment_imgs(SEGMENT_DIR, (WIDTH, HEIGHT))
RW_PRIMITIVES_IMGS = get_segment_imgs(RW_PRIMITIVES_DIR, (WIDTH, HEIGHT))
PRIMITIVE_IMGS = get_primitive_imgs(CONTENT_DIR, (WIDTH, HEIGHT))

In [ ]:
def loop(es, max_val, primitives_selected, prompt_embedding, predictN, prompt_text, report_interval, gamma=2):
  # Check if the directory exists
  if not path.exists(path.join(RESULTS_DIR, prompt_text)):
    os.makedirs(path.join(RESULTS_DIR, prompt_text))
  
  with open(path.join(RESULTS_DIR, prompt_text, 'results.json'), 'w') as f:
    pass # Open to resent the file

  i = 0
  while not es.stop():
      a = time.time()
      solutions = es.ask()
      rasterized_imgs = []

      b = time.time()
      for it, solution in enumerate(solutions):
        # Reshape (one for each primitive)
        solution = solution.reshape(-1, predictN)

        solution = clip_sol(solution, max_val)
        

        img = rasterize_shapes_2(primitives_selected, solution, (WIDTH, HEIGHT))
        rasterized_imgs.append(Image.alpha_composite(
                                  #Image.new("RGBA", img.size, (54, 108, 176, 255)),  # Blue background
                                  Image.new("RGBA", img.size, (255, 255, 255, 255)),  # White background
                                  img
                              ).convert("RGB"))

      rasterized_imgs = np.stack(rasterized_imgs, axis=0) # (population_size, 300, 300, 4)
      rasterized_imgs = torch.tensor(rasterized_imgs).permute(0, 3, 1, 2).to(DEVICE)
      rasterized_emb = CLIP_emb_from_IMG(rasterized_imgs) # (population_size, 512)

      losses = torch.nn.functional.cosine_similarity(rasterized_emb, prompt_embedding, dim=1)

      # 1 minus cosine_similarity because we want to maximize the similarity
      # **gamma (right now **2) to emphasize small differences in the loss
      es.tell(solutions, [(1 - l.item())**gamma for l in losses])

      # Save loss info (aka fitness)
      with open(path.join(RESULTS_DIR, prompt_text, 'results.json'), 'a') as f:
        json.dump({i: losses.cpu().numpy().tolist()}, f)
        f.write('\n')

      
      # Update epoch
      i += 1

      if i % report_interval == 0:
        img = rasterized_imgs[losses.argmax()].cpu().numpy()
        img = np.transpose(img, (1,2,0))
        img_pil = Image.fromarray(img.astype('uint8'), 'RGB')

        # Save best img
        img_pil.save(path.join(RESULTS_DIR, prompt_text, f'epoch_{i}.png'))
        clear_output(wait=True)  # Clear previous output
        display(img_pil)         # Display the new image
    


In [ ]:

sigma = 0.1
maxiter = 500
popsize = 32
max_val = 0.5
predictN = 5

# Execute the loop for each prompt
for prompt_text, elems in RW_PRIMITIVES_IMGS.items():
    x0 = np.ones((len(elems), predictN)) * 0.5
    x0[0, 3] = -0.5 # No rotation

    es = cma.CMAEvolutionStrategy(x0.flatten(), sigma, {
            "maxiter": maxiter,
            "popsize": popsize,
            "bounds": [-max_val, max_val],
    })
    
    
    loop(
        es=es, 
        max_val=max_val,
        primitives_selected=list(elems.values()),
        prompt_embedding=CLIP_emb_from_TEXT(prompt_text),
        predictN=predictN,
        prompt_text=prompt_text,
        report_interval=15,
        gamma=2
    )

(16_w,32)-aCMA-ES (mu_w=9.2,w_1=19%) in dimension 15 (seed=365668, Sat Mar 29 19:37:44 2025)


KeyboardInterrupt: 